# Phase Coordinates Demo

This notebook demonstrates `cycle_by_cycle_pca_coordinates` on synthetic 3D
noisy cyclic data.

**Workflow**
1. Generate 3D noisy cyclic data (a circle tilted in 3-D space, corrupted by
   Gaussian noise).
2. Run `cycle_by_cycle_pca_coordinates` to obtain, for each time point:
   - The **cycle** index and **phase** within the cycle.
   - The **radius** in the local PCA plane.
   - The **perpendicular deviation** out of that plane.
3. Reconstruct the original data from the recovered PCA coordinates and verify
   the match.


## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from phase_coordinates import cycle_by_cycle_pca_coordinates


## 2 · Generate synthetic 3D noisy cyclic data

In [ ]:
rng = np.random.default_rng(42)

n_cycles          = 8
samples_per_cycle = 120
noise_std         = 0.08
radius            = 1.5
tilt_angle        = np.pi / 5   # tilt of the circle plane around x-axis

n_time = n_cycles * samples_per_cycle
fs     = float(samples_per_cycle)  # 1 Hz cycle frequency

t          = np.arange(n_time) / fs
phase_true = 2 * np.pi * t        # unwrapped true phase

# Clean circular trajectory in a tilted plane
u_clean = radius * np.cos(phase_true)
v_clean = radius * np.sin(phase_true)

X_clean = np.column_stack([
    u_clean,
    v_clean * np.cos(tilt_angle),
    v_clean * np.sin(tilt_angle),
])

X_noisy = X_clean + rng.standard_normal(X_clean.shape) * noise_std

print(f"Data shape : {X_noisy.shape}  (n_time x 3 features)")
print(f"Noise STD  : {noise_std}  |  True radius : {radius}")


## 3 · Run cycle-by-cycle PCA

In [ ]:
coords, models = cycle_by_cycle_pca_coordinates(
    X_noisy,
    phase=phase_true,   # supply pre-computed unwrapped phase
)

print(coords.head(10))
print(f"\nDetected {len(models)} cycles.")


## 4 · Inspect the recovered coordinates

For each time point we have:
- **`phase_in_cycle`** – where in the cycle we are (0 to 2pi).
- **`radius_local`** – distance from the cycle centre in the local PCA plane.
- **`perp_local`** – deviation perpendicular to that plane.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

cycle_color = coords["cycle"].to_numpy()
colors = plt.cm.tab10(cycle_color % 10)

for ax, col, ylabel in zip(
    axes,
    ["radius_local", "theta_local_wrapped", "perp_local"],
    ["Radius in PCA plane", "Angle in PCA plane (rad)", "Perp. deviation"],
):
    ax.scatter(t, coords[col], c=colors, s=2, alpha=0.7)
    ax.set_ylabel(ylabel)
    ax.axhline(0, color="k", lw=0.5)

axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Cycle-by-cycle PCA coordinates (colour = cycle index)")
plt.tight_layout()
plt.show()


## 5 · Phase-plane portrait (cycle by cycle)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for cyc, m in models.items():
    idx = m["indices"]
    ax.plot(
        coords.loc[idx, "pc1_local"],
        coords.loc[idx, "pc2_local"],
        lw=0.8,
        alpha=0.7,
        label=f"cycle {cyc}",
    )

ax.set_xlabel("PC1 (local)")
ax.set_ylabel("PC2 (local)")
ax.set_title("Local PCA phase plane - all cycles")
ax.set_aspect("equal")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 6 · Reconstruct X from PCA coordinates and verify recovery

In [ ]:
X_rec = np.full_like(X_noisy, np.nan)

for cyc, m in models.items():
    idx    = m["indices"]
    center = m["center"]      # shape (3,)
    comps  = m["components"]  # shape (3, 3) - rows are PC axes
    p1 = coords.loc[idx, "pc1_local"].to_numpy()
    p2 = coords.loc[idx, "pc2_local"].to_numpy()
    p3 = coords.loc[idx, "pc3_local"].to_numpy()
    X_rec[idx] = (
        p1[:, None] * comps[0]
        + p2[:, None] * comps[1]
        + p3[:, None] * comps[2]
        + center
    )

valid = ~np.isnan(X_rec).any(axis=1)
max_error = np.abs(X_noisy[valid] - X_rec[valid]).max()
print(f"Max reconstruction error : {max_error:.2e}  (should be near machine epsilon)")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

for i, (ax, lbl) in enumerate(zip(axes, ["X", "Y", "Z"])):
    ax.plot(t[valid], X_noisy[valid, i], lw=0.8, label="Original (noisy)", alpha=0.6)
    ax.plot(t[valid], X_rec[valid, i],   lw=1.5, label="Reconstructed",    ls="--")
    ax.set_ylabel(lbl)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Original vs Reconstructed data (should overlap exactly)")
plt.tight_layout()
plt.show()


## 7 · 3D visualisation

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection="3d")

ax.plot(*X_clean.T,           lw=1.5, color="steelblue", alpha=0.4, label="Clean")
ax.scatter(*X_noisy[valid].T, s=2, c=t[valid], cmap="viridis",  alpha=0.6, label="Noisy")
ax.scatter(*X_rec[valid].T,   s=6, color="crimson",              alpha=0.5, label="Reconstructed")

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D cyclic data: clean, noisy, reconstructed")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Summary

| Quantity | Description |
|---|---|
| `radius_local` | Radius in the local PCA phase plane (reflects movement amplitude) |
| `theta_local_wrapped` | Instantaneous angle in the phase plane |
| `perp_local` (= `pc3_local`) | Deviation perpendicular to the phase plane |

The reconstruction error is near machine precision, confirming that
`(pc1_local, pc2_local, pc3_local)` together with the per-cycle `center` and
`components` fully encode the original noisy data.
